# LightGBM + XGBoost + CatBoost 앙상블 모델링 v1

## 구성
| 단계 | 내용 |
|------|------|
| 1 | 데이터 로드 & v2 피처 엔지니어링 |
| 2 | LightGBM 학습 (품목별, Optuna 튜닝) |
| 3 | XGBoost 학습 (품목별, Optuna 튜닝) |
| 4 | CatBoost 학습 (품목별, Optuna 튜닝) |
| 5 | 3모델 앙상블 가중치 최적화 |
| 6 | Test 예측 (자기회귀 3단계) & Submission 저장 |

**평가 지표**: MAE  
**검증 방식**: 품목별 마지막 9순(3개월) 시계열 holdout  
**앙상블**: LGB + XGB + CatBoost (Optuna 품목별 최적 가중치)

In [16]:
pip install catboost

In [17]:
import numpy as np
import pandas as pd
import glob, os, warnings
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

# ── 설정 ─────────────────────────────────────────────────────
TRAIN_PATH = 'train_target_only_cleaned (1).csv'
TEST_DIR   = 'test'
SAVE_PATH  = 'ensemble_lgb_xgb_cat_submission.csv'

USE_LOG    = False   # True → log1p 변환 타깃
VALID_SIZE = 9       # Optuna / EarlyStopping 용 validation (9순 = 3개월)
TEST_SIZE  = 9       # 로컬 NMAE 평가 전용 holdout (절대 학습에 사용 안 함)
N_TRIALS   = 30      # Optuna 탐색 횟수 (품목당)
W_TRIALS   = 30      # 앙상블 가중치 탐색 횟수

GROUP_COLS = ['품목명', '품종명', '거래단위', '등급']
CAT_COLS   = ['품종명', '거래단위', '등급']
TARGET_COL = '평균가격(원)'
NORMAL_COL = '평년 평균가격(원)'
TIME_COL   = '시점'
순_map      = {'상순': 0, '중순': 1, '하순': 2}

---
## 1. 데이터 로드 & v2 피처 엔지니어링

In [18]:
df = pd.read_csv(TRAIN_PATH)
print('원본 shape:', df.shape)

# ── 시간 파싱 ─────────────────────────────────────────────────
df['year']     = df[TIME_COL].str[:4].astype(int)
df['month']    = df[TIME_COL].str[4:6].astype(int)
df['순_str']   = df[TIME_COL].str[6:]
df['순_num']   = df['순_str'].map(순_map)
df['time_idx'] = df['year'] * 36 + (df['month'] - 1) * 3 + df['순_num']

df = df.sort_values(GROUP_COLS + ['time_idx']).reset_index(drop=True)
df['y'] = np.log1p(df[TARGET_COL]) if USE_LOG else df[TARGET_COL]

print(f'time_idx 범위: {df["time_idx"].min()} ~ {df["time_idx"].max()}')
print(f'연도: {sorted(df["year"].unique())}')
print(f'품목: {sorted(df["품목명"].unique())}')
df.head(3)

원본 shape: (1425, 7)
time_idx 범위: 72648 ~ 72791
연도: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
품목: ['감자', '건고추', '깐마늘(국산)', '대파', '무', '배', '배추', '사과', '상추', '양파']


,시점,품목명,품종명,거래단위,등급,평년 평균가격(원),평균가격(원),year,month,순_str,순_num,time_idx,y
0,201801상순,감자,감자 수미,20키로상자,상,24660.031746,44170.285714,2018,1,상순,0,72648,44170.285714
1,201801중순,감자,감자 수미,20키로상자,상,23299.444444,48283.777778,2018,1,중순,1,72649,48283.777778
2,201801하순,감자,감자 수미,20키로상자,상,25218.007407,50243.000000,2018,1,하순,2,72650,50243.000000


In [19]:
g = df.groupby(GROUP_COLS)['y']

# ── [기존] Lag 1~3 ────────────────────────────────────────────
for lag in [1, 2, 3]:
    df[f'lag{lag}'] = g.shift(lag)

df['diff1']  = df['lag1'] - df['lag2']
df['diff2']  = df['lag2'] - df['lag3']
df['ratio1'] = df['lag1'] / (df['lag2'] + 1e-6)
df['ratio2'] = df['lag2'] / (df['lag3'] + 1e-6)

df['rolling_mean_3'] = g.transform(lambda x: x.shift(1).rolling(3).mean())
df['sin_month']      = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month']      = np.cos(2 * np.pi * df['month'] / 12)
df['normal_ratio']   = (df[TARGET_COL] - df[NORMAL_COL]) / (df[NORMAL_COL] + 1e-6)

# ── [추가] 중기·전년 Lag ──────────────────────────────────────
for lag in [6, 12, 36]:
    df[f'lag{lag}'] = g.shift(lag)

df['yoy_change'] = (df['lag1'] - df['lag36']) / (df['lag36'] + 1e-6)

# ── [추가] 롤링·EWM ──────────────────────────────────────────
df['rolling_mean_6'] = g.transform(lambda x: x.shift(1).rolling(6).mean())
df['rolling_std_3']  = g.transform(lambda x: x.shift(1).rolling(3).std())
df['rolling_std_6']  = g.transform(lambda x: x.shift(1).rolling(6).std())
df['ewm_span6']      = g.transform(lambda x: x.shift(1).ewm(span=6, adjust=False).mean())

# ── [추가] 변동성·모멘텀 ─────────────────────────────────────
_rm6 = g.transform(lambda x: x.shift(1).rolling(6).mean())
_rs6 = g.transform(lambda x: x.shift(1).rolling(6).std())
df['rolling_cv6']    = _rs6 / (_rm6 + 1e-6)
df['price_momentum'] = df['lag1'] / (df['lag3'] + 1e-6) - 1

_item_mean = df.groupby('품목명')['y'].transform('mean')
_item_std  = df.groupby('품목명')['y'].transform('std')
df['price_zscore'] = (df['lag1'] - _item_mean) / (_item_std + 1e-6)

# ── [추가] 시간·이벤트 ───────────────────────────────────────
df['sun_sin']    = np.sin(2 * np.pi * df['순_num'] / 3)
df['sun_cos']    = np.cos(2 * np.pi * df['순_num'] / 3)
df['year_trend'] = df['year'] - 2018
df['is_2020']    = (df['year'] == 2020).astype(int)

# ── [추가] 평년 대비 확장 ────────────────────────────────────
g_nr = df.groupby(GROUP_COLS)['normal_ratio']
df['normal_ratio_lag1']  = g_nr.shift(1)
df['normal_ratio_roll3'] = g_nr.transform(lambda x: x.shift(1).rolling(3).mean())
df['is_above_normal']    = (df['normal_ratio'] > 0).astype(int)
df['log_normal_price']   = np.log1p(df[NORMAL_COL])

print('피처 엔지니어링 완료, shape:', df.shape)

피처 엔지니어링 완료, shape: (1425, 43)


In [20]:
# ── 피처 목록 ────────────────────────────────────────────────
FEATURE_COLS = [
    '품종명', '거래단위', '등급',
    'lag1', 'lag2', 'lag3',
    'diff1', 'diff2', 'ratio1', 'ratio2',
    'rolling_mean_3',
    'month', '순_num', 'sin_month', 'cos_month', 'time_idx',
    'normal_ratio', 'log_normal_price',
    'lag6', 'lag12', 'lag36', 'yoy_change',
    'rolling_mean_6', 'rolling_std_3', 'rolling_std_6', 'ewm_span6',
    'rolling_cv6', 'price_momentum', 'price_zscore',
    'sun_sin', 'sun_cos', 'year_trend', 'is_2020',
    'normal_ratio_lag1', 'normal_ratio_roll3', 'is_above_normal',
]

# LGB용: category dtype
df_lgb = df.dropna(subset=['lag1', 'lag2', 'lag3']).copy()
for c in CAT_COLS:
    df_lgb[c] = df_lgb[c].astype('category')

# XGB용: LabelEncoder
le_dict = {}
df_xgb  = df.dropna(subset=['lag1', 'lag2', 'lag3']).copy()
for c in CAT_COLS:
    le = LabelEncoder()
    df_xgb[c] = le.fit_transform(df_xgb[c].astype(str))
    le_dict[c] = le

# CAT용: 문자열 그대로 (CatBoost가 cat_features 인덱스로 처리)
df_cat    = df.dropna(subset=['lag1', 'lag2', 'lag3']).copy()
cat_feat_idx = [FEATURE_COLS.index(c) for c in CAT_COLS]

# 공통 df_model (LGB 기준으로 보관)
df_model = df_lgb.copy()

# 품목별 통계 (test zscore 계산용)
item_stats = (
    df.groupby('품목명')['y']
    .agg(['mean', 'std'])
    .to_dict('index')
)

print(f'학습 데이터: {df_model.shape[0]}행 × {len(FEATURE_COLS)}개 피처')
print(f'카테고리 피처 인덱스 (CatBoost용): {cat_feat_idx}')

학습 데이터: 1392행 × 36개 피처
카테고리 피처 인덱스 (CatBoost용): [0, 1, 2]


---
## 2. LightGBM 학습 (Optuna 하이퍼파라미터 튜닝)

In [21]:
def three_way_split(item_df, valid_size=VALID_SIZE, test_size=TEST_SIZE):
    """
    tr  : 학습
    va  : Optuna / EarlyStopping 용 validation
    te  : 로컬 NMAE 평가 전용 (학습/튜닝에 절대 사용 안 함)
    """
    item_df = item_df.sort_values('time_idx').reset_index(drop=True)
    if len(item_df) <= valid_size + test_size:
        return None, None, None
    tr = item_df.iloc[:-(valid_size + test_size)].copy()
    va = item_df.iloc[-(valid_size + test_size):-test_size].copy()
    te = item_df.iloc[-test_size:].copy()
    return tr, va, te


def tune_lgb(item, n_trials=N_TRIALS):
    item_df = df_lgb[df_lgb['품목명'] == item].copy()
    tr, va, _ = three_way_split(item_df)
    if tr is None:
        return None, None

    X_tr, y_tr = tr[FEATURE_COLS], tr['y']
    X_va, y_va = va[FEATURE_COLS], va['y']

    def objective(trial):
        params = {
            'objective'        : 'regression',
            'metric'           : 'mae',
            'verbosity'        : -1,
            'n_estimators'     : trial.suggest_int('n_estimators', 300, 1500),
            'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves'       : trial.suggest_int('num_leaves', 16, 128),
            'max_depth'        : trial.suggest_int('max_depth', 3, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'random_state'     : 42,
        }
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_va, y_va)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(-1)])
        return mean_absolute_error(y_va, model.predict(X_va))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best  = study.best_params
    model = LGBMRegressor(**best, objective='regression', metric='mae',
                          verbosity=-1, random_state=42)
    model.fit(X_tr, y_tr,
              eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                          lgb.log_evaluation(-1)])
    return model, mean_absolute_error(y_va, model.predict(X_va))


lgb_models, lgb_val_maes = {}, {}
for item in sorted(df_lgb['품목명'].unique()):
    model, mae = tune_lgb(item)
    if model is None:
        print(f'[LGB] {item} → 데이터 부족, skip'); continue
    lgb_models[item]   = model
    lgb_val_maes[item] = mae
    print(f'[LGB] {item:15s} | val MAE: {mae:>10,.1f}')

print(f'\n[LGB] 전체 평균 val MAE: {np.mean(list(lgb_val_maes.values())):,.1f}')

[LGB] 감자              | val MAE:      746.4
[LGB] 건고추             | val MAE:   28,044.0
[LGB] 깐마늘(국산)         | val MAE:    7,074.9
[LGB] 대파              | val MAE:      109.2
[LGB] 무               | val MAE:      189.3
[LGB] 배               | val MAE:    2,952.8
[LGB] 배추              | val MAE:    1,100.0
[LGB] 사과              | val MAE:    1,105.9
[LGB] 상추              | val MAE:      168.4
[LGB] 양파              | val MAE:       29.9

[LGB] 전체 평균 val MAE: 4,152.1


---
## 3. XGBoost 학습 (Optuna 하이퍼파라미터 튜닝)

XGBoost는 category dtype 미지원 → LabelEncoder 인코딩 후 Optuna 탐색.

In [22]:
def tune_xgb(item, n_trials=N_TRIALS):
    item_df = df_xgb[df_xgb['품목명'] == item].copy()
    tr, va, _ = three_way_split(item_df)
    if tr is None:
        return None, None

    X_tr, y_tr = tr[FEATURE_COLS], tr['y']
    X_va, y_va = va[FEATURE_COLS], va['y']

    def objective(trial):
        params = {
            'objective'       : 'reg:absoluteerror',
            'tree_method'     : 'hist',
            'verbosity'       : 0,
            'random_state'    : 42,
            'n_estimators'    : trial.suggest_int('n_estimators', 300, 1500),
            'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth'       : trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
            'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        }
        model = XGBRegressor(**params, early_stopping_rounds=50)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return mean_absolute_error(y_va, model.predict(X_va))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best  = study.best_params
    model = XGBRegressor(**best, objective='reg:absoluteerror', tree_method='hist',
                         verbosity=0, random_state=42, early_stopping_rounds=50)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return model, mean_absolute_error(y_va, model.predict(X_va))


xgb_models, xgb_val_maes = {}, {}
for item in sorted(df_xgb['품목명'].unique()):
    model, mae = tune_xgb(item)
    if model is None:
        print(f'[XGB] {item} → 데이터 부족, skip'); continue
    xgb_models[item]   = model
    xgb_val_maes[item] = mae
    print(f'[XGB] {item:15s} | val MAE: {mae:>10,.1f}')

print(f'\n[XGB] 전체 평균 val MAE: {np.mean(list(xgb_val_maes.values())):,.1f}')

[XGB] 감자              | val MAE:      637.6
[XGB] 건고추             | val MAE:   18,162.8
[XGB] 깐마늘(국산)         | val MAE:    8,371.4
[XGB] 대파              | val MAE:      135.4
[XGB] 무               | val MAE:      120.3


[W 2026-03-25 19:11:16,526] Trial 23 failed with parameters: {'n_estimators': 1495, 'learning_rate': 0.059547322831296365, 'max_depth': 5, 'min_child_weight': 10, 'subsample': 0.8491032300233923, 'colsample_bytree': 0.7373084382798768, 'reg_alpha': 0.0029144135116962438, 'reg_lambda': 0.13921007961737492} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\user\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_32632\2103600113.py", line 26, in objective
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\user\anaconda3\Lib\site-packages\xgboost\core.py", line 729, in inner_f
    return func(**kwargs)
  File "c:\Users\user\anaconda3\Lib\site-packages\xgboost\sklearn.py", line 1247, in fit
    self._Booster = train(
           

KeyboardInterrupt: 

---
## 4. CatBoost 학습 (Optuna 하이퍼파라미터 튜닝)

CatBoost는 카테고리 피처를 **인코딩 없이** 직접 처리 → 원본 문자열 그대로 사용.

In [ ]:
def tune_cat(item, n_trials=N_TRIALS):
    item_df = df_cat[df_cat['품목명'] == item].copy()
    tr, va, _ = three_way_split(item_df)
    if tr is None:
        return None, None

    X_tr, y_tr = tr[FEATURE_COLS], tr['y']
    X_va, y_va = va[FEATURE_COLS], va['y']

    for c in CAT_COLS:
        X_tr[c] = X_tr[c].astype(str)
        X_va[c] = X_va[c].astype(str)

    def objective(trial):
        params = {
            'loss_function'      : 'MAE',
            'eval_metric'        : 'MAE',
            'random_seed'        : 42,
            'verbose'            : 0,
            'cat_features'       : CAT_COLS,
            'iterations'         : trial.suggest_int('iterations', 300, 1500),
            'learning_rate'      : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth'              : trial.suggest_int('depth', 3, 10),
            'l2_leaf_reg'        : trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_strength'    : trial.suggest_float('random_strength', 0.0, 1.0),
            'border_count'       : trial.suggest_int('border_count', 32, 255),
        }
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va),
                  early_stopping_rounds=50, verbose=0)
        return mean_absolute_error(y_va, model.predict(X_va))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best  = study.best_params
    model = CatBoostRegressor(**best, loss_function='MAE', eval_metric='MAE',
                               random_seed=42, verbose=0, cat_features=CAT_COLS)
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va),
              early_stopping_rounds=50, verbose=0)
    return model, mean_absolute_error(y_va, model.predict(X_va))


cat_models, cat_val_maes = {}, {}
for item in sorted(df_cat['품목명'].unique()):
    model, mae = tune_cat(item)
    if model is None:
        print(f'[CAT] {item} → 데이터 부족, skip'); continue
    cat_models[item]   = model
    cat_val_maes[item] = mae
    print(f'[CAT] {item:15s} | val MAE: {mae:>10,.1f}')

print(f'\n[CAT] 전체 평균 val MAE: {np.mean(list(cat_val_maes.values())):,.1f}')

---
## 5. 3모델 앙상블 가중치 최적화

최종 예측 = w_lgb × LGB + w_xgb × XGB + w_cat × CatBoost (합계=1, Optuna 최적화)


In [ ]:
best_weights = {}   # {품목명: (w_lgb, w_xgb, w_cat)}

for item in sorted(lgb_models.keys()):
    if item not in xgb_models or item not in cat_models:
        continue

    # 가중치 최적화는 validation(va) 기준 — test(te)는 절대 사용 안 함
    _, va_lgb, _ = three_way_split(df_lgb[df_lgb['품목명'] == item].copy())
    _, va_xgb, _ = three_way_split(df_xgb[df_xgb['품목명'] == item].copy())
    _, va_cat, _ = three_way_split(df_cat[df_cat['품목명'] == item].copy())
    for c in CAT_COLS:
        va_cat[c] = va_cat[c].astype(str)

    p_lgb = lgb_models[item].predict(va_lgb[FEATURE_COLS])
    p_xgb = xgb_models[item].predict(va_xgb[FEATURE_COLS])
    p_cat = cat_models[item].predict(va_cat[FEATURE_COLS])
    y_va  = va_lgb['y'].values

    def weight_objective(trial):
        w_l = trial.suggest_float('w_lgb', 0.0, 1.0)
        w_x = trial.suggest_float('w_xgb', 0.0, 1.0 - w_l)
        w_c = 1.0 - w_l - w_x
        return mean_absolute_error(y_va, w_l * p_lgb + w_x * p_xgb + w_c * p_cat)

    w_study = optuna.create_study(direction='minimize')
    w_study.optimize(weight_objective, n_trials=W_TRIALS, show_progress_bar=False)
    w_l = w_study.best_params['w_lgb']
    w_x = w_study.best_params['w_xgb']
    best_weights[item] = (w_l, w_x, 1.0 - w_l - w_x)

print('가중치 최적화 완료')
for item, (w_l, w_x, w_c) in sorted(best_weights.items()):
    print(f'  {item:12s} | LGB {w_l:.2f}  XGB {w_x:.2f}  CAT {w_c:.2f}')

---
## 6. Test 예측 (자기회귀 3단계 앙상블)

각 테스트 파일(TEST_00 ~ TEST_24)에 대해:
1. T-8순 ~ T+0순 이력 + 학습 데이터 이력 결합
2. T+1, T+2, T+3 순서로 자기회귀 예측
3. LGB / XGB / CatBoost 예측값을 품목별 최적 가중치로 합산

In [ ]:
df_hist = df.copy()

def parse_test_rel(시점_str):
    s = str(시점_str).strip()
    if 'T+' in s: return  int(s.replace('T+', '').replace('순', ''))
    if 'T-' in s: return -int(s.replace('T-', '').replace('순', ''))
    return 0

def advance_time(year, month, sun):
    sun += 1
    if sun == 3:
        sun, month = 0, month + 1
    if month == 13:
        month, year = 1, year + 1
    return year, month, sun

def compute_ewm(history, span=6):
    if not history: return np.nan
    alpha, ewm = 2 / (span + 1), history[0]
    for v in history[1:]: ewm = alpha * v + (1 - alpha) * ewm
    return ewm

def build_feature_row(history, normal_price, year, month, sun_num,
                      품종명, 거래단위, 등급, item, mode='lgb'):
    """mode: 'lgb' | 'xgb' | 'cat'"""
    n   = len(history)
    lag = lambda k: history[-k] if n >= k else np.nan
    lag1, lag2, lag3   = lag(1), lag(2), lag(3)
    lag6, lag12, lag36 = lag(6), lag(12), lag(36)

    safe_mean = lambda lst: float(np.nanmean(lst)) if lst else np.nan
    safe_std  = lambda lst: float(np.nanstd(lst))  if len(lst) > 1 else np.nan
    r3  = history[-3:] if n >= 3 else history
    r6  = history[-6:] if n >= 6 else history
    rm3, rm6 = safe_mean(r3), safe_mean(r6)
    rs3, rs6 = safe_std(r3),  safe_std(r6)
    cv6      = rs6 / (rm6 + 1e-6)
    ewm6     = compute_ewm(history)
    momentum = lag1 / (lag3 + 1e-6) - 1
    yoy      = (lag1 - lag36) / (lag36 + 1e-6) if not np.isnan(lag36) else np.nan
    i_s      = item_stats.get(item, {'mean': 0, 'std': 1})
    zscore   = (lag1 - i_s['mean']) / (i_s['std'] + 1e-6)

    sin_m = np.sin(2 * np.pi * month   / 12)
    cos_m = np.cos(2 * np.pi * month   / 12)
    sin_s = np.sin(2 * np.pi * sun_num / 3)
    cos_s = np.cos(2 * np.pi * sun_num / 3)

    lag1_raw = float(np.expm1(lag1)) if USE_LOG else lag1
    lag2_raw = float(np.expm1(lag2)) if USE_LOG else lag2
    nr       = (lag1_raw - normal_price) / (normal_price + 1e-6)
    nr_lag1  = (lag2_raw - normal_price) / (normal_price + 1e-6) if not np.isnan(lag2) else np.nan
    nr_r3    = [(float(np.expm1(h)) if USE_LOG else h - normal_price) / (normal_price + 1e-6) for h in r3]

    row = {
        '품종명': 품종명, '거래단위': 거래단위, '등급': 등급,
        'lag1': lag1, 'lag2': lag2, 'lag3': lag3,
        'diff1': lag1 - lag2 if not (np.isnan(lag1) or np.isnan(lag2)) else np.nan,
        'diff2': lag2 - lag3 if not (np.isnan(lag2) or np.isnan(lag3)) else np.nan,
        'ratio1': lag1 / (lag2 + 1e-6), 'ratio2': lag2 / (lag3 + 1e-6),
        'rolling_mean_3': rm3, 'month': month, '순_num': sun_num,
        'sin_month': sin_m, 'cos_month': cos_m,
        'time_idx': year * 36 + (month - 1) * 3 + sun_num,
        'normal_ratio': nr, 'log_normal_price': np.log1p(normal_price),
        'lag6': lag6, 'lag12': lag12, 'lag36': lag36, 'yoy_change': yoy,
        'rolling_mean_6': rm6, 'rolling_std_3': rs3, 'rolling_std_6': rs6,
        'ewm_span6': ewm6, 'rolling_cv6': cv6,
        'price_momentum': momentum, 'price_zscore': zscore,
        'sun_sin': sin_s, 'sun_cos': cos_s,
        'year_trend': year - 2018, 'is_2020': int(year == 2020),
        'normal_ratio_lag1': nr_lag1,
        'normal_ratio_roll3': safe_mean(nr_r3),
        'is_above_normal': int(nr > 0),
    }

    X = pd.DataFrame([row])
    if mode == 'lgb':
        for c in CAT_COLS: X[c] = X[c].astype('category')
    elif mode == 'xgb':
        for c in CAT_COLS:
            val = str(X[c].iloc[0])
            X[c] = le_dict[c].transform([val])[0] if val in le_dict[c].classes_ else -1
    else:  # cat
        for c in CAT_COLS: X[c] = X[c].astype(str)
    return X[FEATURE_COLS]

In [ ]:
def predict_test_file(test_path):
    test    = pd.read_csv(test_path)
    test_id = os.path.splitext(os.path.basename(test_path))[0]
    test['rel_idx'] = test[TIME_COL].apply(parse_test_rel)
    test = test.sort_values('rel_idx')

    pred_by_step = {1: {}, 2: {}, 3: {}}

    for keys, grp in test.groupby(GROUP_COLS):
        품목명_k, 품종명_k, 거래단위_k, 등급_k = keys
        grp = grp.sort_values('rel_idx').reset_index(drop=True)

        normal_price = float(grp[NORMAL_COL].iloc[-1])
        test_hist_y  = grp[TARGET_COL].tolist()

        mask = (
            (df_hist['품목명']   == 품목명_k) &
            (df_hist['품종명']   == 품종명_k) &
            (df_hist['거래단위'] == 거래단위_k) &
            (df_hist['등급']     == 등급_k)
        )
        train_grp = df_hist[mask].sort_values('time_idx')
        if len(train_grp) == 0:
            continue

        last   = train_grp.iloc[-1]
        cy, cm, cs = advance_time(int(last['year']), int(last['month']), int(last['순_num']))

        train_y  = train_grp['y'].tolist()
        test_y   = [np.log1p(v) if USE_LOG else v for v in test_hist_y]
        current  = train_y + test_y

        w_l, w_x, w_c = best_weights.get(품목명_k, (1/3, 1/3, 1/3))

        for step in range(1, 4):
            X_lgb = build_feature_row(current, normal_price, cy, cm, cs,
                                      품종명_k, 거래단위_k, 등급_k, 품목명_k, mode='lgb')
            X_xgb = build_feature_row(current, normal_price, cy, cm, cs,
                                      품종명_k, 거래단위_k, 등급_k, 품목명_k, mode='xgb')
            X_cat = build_feature_row(current, normal_price, cy, cm, cs,
                                      품종명_k, 거래단위_k, 등급_k, 품목명_k, mode='cat')

            p_l = float(max(0.0, lgb_models[품목명_k].predict(X_lgb)[0])) if 품목명_k in lgb_models else 0.0
            p_x = float(max(0.0, xgb_models[품목명_k].predict(X_xgb)[0])) if 품목명_k in xgb_models else 0.0
            p_c = float(max(0.0, cat_models[품목명_k].predict(X_cat)[0])) if 품목명_k in cat_models else 0.0

            p_ens = w_l * p_l + w_x * p_x + w_c * p_c
            p_raw = float(np.expm1(p_ens) if USE_LOG else p_ens)

            pred_by_step[step].setdefault(품목명_k, []).append(p_raw)
            current.append(p_ens)
            cy, cm, cs = advance_time(cy, cm, cs)

    all_items = sorted(test['품목명'].unique())
    rows = []
    for step in [1, 2, 3]:
        row = {'시점': f'{test_id}+{step}순'}
        for item in all_items:
            vals = pred_by_step[step].get(item, [0.0])
            row[item] = float(np.mean(vals)) if vals else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

---
## 7. Submission 생성

In [ ]:
sample_sub = pd.read_csv('sample_submission.csv')
test_files = sorted(glob.glob(os.path.join(TEST_DIR, 'TEST_*.csv')))
print(f'Test 파일 수: {len(test_files)}개\n')

pred_dfs = []
for tf in test_files:
    pred_dfs.append(predict_test_file(tf))
    print(f'완료: {os.path.basename(tf)}')

submission = pd.concat(pred_dfs, ignore_index=True)
submission = sample_sub[['시점']].merge(submission, on='시점', how='left')
for col in submission.columns[1:]:
    submission[col] = submission[col].fillna(0).clip(lower=0)
for col in sample_sub.columns:
    if col not in submission.columns:
        submission[col] = 0
submission = submission[sample_sub.columns]

submission.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print(f'\n저장 완료: {SAVE_PATH}  ({submission.shape[0]}행 × {submission.shape[1]}열)')
submission.head(10)

Test 파일 수: 25개

완료: TEST_00.csv
완료: TEST_01.csv
완료: TEST_02.csv
완료: TEST_03.csv
완료: TEST_04.csv
완료: TEST_05.csv
완료: TEST_06.csv
완료: TEST_07.csv
완료: TEST_08.csv
완료: TEST_09.csv
완료: TEST_10.csv
완료: TEST_11.csv
완료: TEST_12.csv
완료: TEST_13.csv
완료: TEST_14.csv
완료: TEST_15.csv
완료: TEST_16.csv
완료: TEST_17.csv
완료: TEST_18.csv
완료: TEST_19.csv
완료: TEST_20.csv
완료: TEST_21.csv
완료: TEST_22.csv
완료: TEST_23.csv
완료: TEST_24.csv

저장 완료: ensemble_lgb_xgb_cat_submission.csv  (75행 × 11열)


,시점,감자,건고추,깐마늘(국산),대파,무,배추,사과,상추,양파,배
0,TEST_00+1순,42243.965864,603866.371321,157211.764970,1634.728087,23314.232829,15727.671717,25708.661176,1154.627550,1333.509946,31047.639220
1,TEST_00+2순,44077.299845,589546.013055,155392.604386,1435.865075,22641.507740,15611.243418,25431.184842,1129.786379,1313.482842,31083.978384
2,TEST_00+3순,43607.168541,579555.642321,155183.251521,1298.098314,21759.282929,15239.241697,25229.488133,1164.281962,1327.327141,30461.927101
3,TEST_01+1순,43334.403286,596062.582652,157004.589524,1927.455678,13453.823021,5163.905020,25418.543367,1013.419768,1312.273187,29691.061698
4,TEST_01+2순,43485.756499,578314.529347,155163.996241,1961.545273,13049.119949,5039.166333,25530.659789,1033.683474,1302.223144,30387.937951
5,TEST_01+3순,42148.913924,568621.865619,155177.361691,2044.806871,13130.836347,5282.643420,25595.188385,1072.577903,1319.196635,30253.054107
6,TEST_02+1순,50829.572983,559875.671487,157992.365096,1228.189991,9833.395625,10865.703378,25660.873961,1236.448185,579.486893,35793.920290
7,TEST_02+2순,44466.907752,562786.993812,155274.341016,1223.919614,9939.013388,12849.333231,25503.648985,1304.632161,605.471398,34645.013371
8,TEST_02+3순,38428.310620,561445.111363,155210.336771,1219.156888,9943.997035,14690.114757,25548.461886,1284.872566,612.890342,34460.187313
9,TEST_03+1순,50826.578797,559962.215115,157228.028369,1472.036111,14260.611760,7830.872151,25684.276092,934.597347,776.130596,35372.478072
